# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# plotting libraries for later
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset: {metadata['name']}")
print(metadata['description'])

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** Since Croissant datasets can have multiple record sets, each uniquely identified by its `@id`. Fields (columns) within each record set are also uniquely identified using their `@id` attribute.

In [ ]:
# List all record sets and their fields using `@id`
# The dataset may define record sets in the metadata JSON under the 'recordSet' key

json_data = dataset.metadata.to_json()
record_sets = json_data.get('recordSet', [])

if len(record_sets) == 0:
    print("No 'recordSet' found directly. Attempting to infer record sets from 'distribution' entries...")
    # Sometimes, datasets embed records by DataDistribution
    record_sets = json_data.get('distribution', [])

all_record_set_ids = []
record_set_fields = {}
for rec in record_sets:
    rec_id = rec['@id'] if isinstance(rec, dict) and '@id' in rec else rec
    all_record_set_ids.append(rec_id)
    print(f"Found record set: {rec_id}")
    # Try to list fields/columns if present
    if isinstance(rec, dict) and 'field' in rec:
        print("  Fields (by @id):")
        field_ids = []
        for field in rec['field']:
            fid = field['@id'] if isinstance(field, dict) else field
            print(f"    {fid}")
            field_ids.append(fid)
        record_set_fields[rec_id] = field_ids
    else:
        record_set_fields[rec_id] = []

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# For demonstration, pick the first available record set id for data extraction
record_sets_to_use = all_record_set_ids
dataframes = {}

for record_set_id in record_sets_to_use:
    print(f"Loading records from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            print(f"Loaded {len(df)} records.")
        else:
            print("Warning: No records found or unable to load records for this record set.")
            df = pd.DataFrame()
    except Exception as e:
        print(f"Error loading records: {str(e)}")
        df = pd.DataFrame()
    dataframes[record_set_id] = df

# Example: Display the first record set's columns and head
if len(record_sets_to_use) > 0:
    first_rs = record_sets_to_use[0]
    print(f"Columns for {first_rs}:\n", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This can include:
- Filtering out values with low abundance (e.g., `Abundance` > 10 if such a field exists).
- Normalizing a numeric column.
- Grouping by a categorical column (e.g., `Sample` or similar), if present.

In [ ]:
# EDA: Select a numeric field for analysis
import numpy as np

# We'll try to automatically guess a numeric field for demonstration
# List numeric columns
numeric_fields = []
df = dataframes[first_rs]
for col in df.columns:
    # Try convert sample values to numeric
    try:
        vals = pd.to_numeric(df[col].dropna(), errors='coerce')
        if vals.notna().sum() > 0:
            unique_vals = vals.unique()
            # Skip IDs and non-numeric fields
            if len(unique_vals) > 1:
                numeric_fields.append(col)
    except Exception:
        continue
print("Numeric fields available:", numeric_fields)

# Demo: pick first numeric field
if numeric_fields:
    numeric_field = numeric_fields[0]  # Use '@id' for clarity
    print(f"Selected numeric field for analysis: {numeric_field}")
else:
    numeric_field = None
    print("No numeric field found for EDA.")

if numeric_field is not None:
    threshold = df[numeric_field].astype(float).mean()  # Use mean as a threshold
    mask = df[numeric_field].astype(float) > threshold
    filtered_df = df[mask].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    col_norm = f"{numeric_field}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, col_norm]].head())

    # Try to group by a categorical variable (if present)
    # Choose first object-type column that's not the numeric_field
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object and df[col].nunique() < 10 and df[col].notna().sum() > 0:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field).agg({numeric_field: 'mean', col_norm: 'mean'})
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df)
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
# Visualization: Histogram and boxplot for selected numeric field
if numeric_field is not None:
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field].astype(float).dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field].astype(float))
    plt.title(f'Boxplot of {numeric_field}')
    plt.xlabel(numeric_field)

    plt.tight_layout()
    plt.show()

    # If group_field is available, visualize the group differences
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field].astype(float))
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR\u00b2 dataset on Mass Spectrometry Analysis of Extracellular Vesicles using `mlcroissant`, explored its schema, extracted records via their record set `@id`, performed filtering and normalization on a numeric field, and visualized data distributions.

Key next steps could involve domain-specific analysis, advanced feature engineering, or training predictive models.

> **All references to record sets and fields in this notebook use their `@id` values for maximum reproducibility and metadata consistency.**